In [ ]:
import itertools as it

import boto3
import botocore
from cliffs_delta import cliffs_delta
from IPython.core.display import display, HTML
from matplotlib import pyplot as plt
from matplotlib import ticker as mpl_ticker
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm


In [ ]:
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-08-15-nobg-pgcomplex-screen"


In [ ]:
df_all = []
for endeavor in [17, *range(20, 29)]:
    for what, prefix in [
        ("nobb", f"endeavor={endeavor}/external-competitions/stage=2+what=collated/"),
        ("selfbb", f"endeavor={endeavor}/external-competitions-focalbb/stage=2+what=collated/"),
        ("truebb", f"endeavor={endeavor}/external-competitions-focalbb-true/stage=2+what=collated/"),
    ]:
        s3_handle = boto3.resource(
            's3',
            region_name="us-east-2",
            config=botocore.config.Config(
                signature_version=botocore.UNSIGNED,
            ),
        )
        bucket_handle = s3_handle.Bucket("prq49-20stint")


        competitions = bucket_handle.objects.filter(Prefix=prefix)
        dfs = [
            pd.read_csv(
            f's3://prq49-20stint/{competitions.key}',
            )
            for competitions in competitions
        ]
        print(len(dfs))
        df = pd.concat(dfs, ignore_index=True)
        df["kind"] = what

        df_all.append(df)

df_ecoselfcontext = pd.concat(df_all, ignore_index=True)


In [ ]:
dfx = df_ecoselfcontext.loc[
    (df_ecoselfcontext["Root ID"] == 0),
    ["Competition Series","Focal Prevalence", "kind"],
]
with tp.teed(
    sns.scatterplot,
    data=dfx.round().groupby(["Competition Series", "kind"]).mean().reset_index().pivot(
        index="Competition Series", columns="kind", values="Focal Prevalence"
    ),
    x="truebb",
    y="selfbb",
    teeplot_subdir=teeplot_subdir,
) as ax:
    ax.plot([0, 1], [0, 1], ls="--", c="gray")  # Add a diagonal reference line


In [ ]:
with tp.teed(
    sns.boxplot,
    data=dfx.groupby(["Competition Series", "kind"]).mean().reset_index().astype(
        {"Competition Series": str},
    ),
    y="Focal Prevalence",
    hue="kind",
    notch=True,
    teeplot_subdir=teeplot_subdir,
) as ax:
    pass


# get data


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket('prq49-20stint')

dfs = []
for endeavor in [17, *range(20, 29)]:
    series_profiles, = bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/series-profiles/stage=8+what=elaborated/',
    )
    df = pd.read_csv(
        f's3://prq49-20stint/{series_profiles.key}',
        compression='xz',
    )
    dfs.append(df)

df_pgcomplex = pd.concat(dfs, ignore_index=True)


# how does fitness complexity change over time?


In [ ]:
x = "Flagged Advantageous Sites"
y = "Cardinal Interface Complexity"

data = df_pgcomplex[[x, y, "Series"]].reset_index().dropna()
data["x_quantile"] = data[x].rank(pct=True)
data["y_quantile"] = data[y].rank(pct=True)
data["score"] = data[["x_quantile", "y_quantile"]].min(axis=1)
dft = data.nlargest(
    20,
    "score",
    keep="all"
)
data["target"] = data.index.isin(dft.index)
dft


In [ ]:
for inner, iqr in it.product(
    (
        "box",
        None,
    ),
    (
        "shade",
        None,
    ),
):
    with tp.teed(
        sns.scatterplot,
        data=data,
        x=x,
        y=y,
        hue="target",
        alpha=0.5,
        clip_on=False,
        legend=False,
        lw=1,
        teeplot_outattrs={
            "inner": inner, "iqr": iqr
        },
        teeplot_subdir=teeplot_subdir,
    ) as ax:
        ax.figure.set_size_inches(1.5, 1.5)
        xlim, ylim = ax.get_xlim()[1], ax.get_ylim()[1]

        q2_x = dft[x].min()
        q2_y = dft[y].min()

        if iqr == "shade":
            # Shade area where x > 2x IQR (vertical span)
            ax.axvspan(
                q2_x,
                ax.get_xlim()[1] * 1.1,
                color="k",
                alpha=0.1,
                zorder=-20,
                linewidth=0
            )
            # # Annotation for X outlier region
            # ax.text(
            #     q2_x * 1.015,
            #     q2_y * 0.5,
            #     r"$>2\times$ IQR",
            #     color="k",
            #     fontsize=7,
            #     ha="left",
            #     va="center",
            #     rotation=90,
            # )

            # Shade area where y > 2x IQR (horizontal span)
            ax.axhspan(
                q2_y,
                ax.get_ylim()[1] * 1.1,
                color="k",
                alpha=0.1,
                zorder=-20,
                linewidth=0
            )
            # # Annotation for Y outlier region
            # ax.text(
            #     q2_x * 0.5,
            #     q2_y,
            #     r"$>2\times$ IQR",
            #     color="k",
            #     fontsize=7,
            #     ha="center",
            #     va="bottom",
            # )

        if inner == "box":
            sns.boxplot(
                data=data,
                x=4,
                y=y,
                ax=ax,
                color="lightgray",
                fliersize=0,
                native_scale=True,
                legend=False,
                width=1.2,
                whis=1.5,
            )
            ax.plot(
                [4, 4],
                [
                   data[y].median(),
                    scipy_stats.iqr(data[y]) * 1 + data[y].quantile(0.5),
                ],
                ls=":",
                color="gray",
                lw=1,
                zorder=-10,
            )
            sns.boxplot(
                data=data,
                x=x,
                y=0.9,
                ax=ax,
                color="lightgray",
                fliersize=0,
                native_scale=True,
                orient="h",
                legend=False,
                width=0.5,
                whis=1.5,
            )
            ax.plot(
                [
                    data[x].median(),
                    scipy_stats.iqr(data[x]) * 1 + data[x].quantile(0.5),
                ],
                [0.9, 0.9],
                ls=":",
                color="gray",
                lw=1,
                zorder=-10,
            )
        # sns.scatterplot(
        #     data=data[data["Series"] == what],
        #     x=x,
        #     y=y,
        #     style="Series",
        #     alpha=0.8,
        #     ax=ax,
        #     clip_on=False,
        #     color="white",
        #     legend=False,
        #     lw=2,
        #     markers=["o"],
        # )
        # sns.scatterplot(
        #     data=data[data["Series"] == what],
        #     x=x,
        #     y=y,
        #     ax=ax,
        #     style="Series",
        #     clip_on=False,
        #     color="tab:blue",
        #     legend=False,
        #     markers=["o"],
        # )
        # sns.scatterplot(
        #     data=data[data["Series"] == what],
        #     x=x,
        #     y=y,
        #     ax=ax,
        #     style="Series",
        #     clip_on=False,
        #     color="white",
        #     legend=False,
        #     markers=["o"],
        #     s=2,
        # )
        ax.set_xlim(left=0, right=xlim)
        ax.set_ylim(bottom=0, top=ylim)
        sns.despine(ax=ax)
        ax.set_ylabel("Phenotype\nComplexity")
        ax.set_xlabel("Genotype Complexity      ")


In [ ]:
data = df_pgcomplex[["Flagged Advantageous Sites", "Cardinal Interface Complexity", "Series"]].reset_index().dropna()
data["x_quantile"] = data["Flagged Advantageous Sites"].rank(pct=True)
data["y_quantile"] = data["Cardinal Interface Complexity"].rank(pct=True)
data["score"] = data[["x_quantile", "y_quantile"]].min(axis=1)

data["Phenotype\nComplexity"] = data["Cardinal Interface Complexity"]
data["Genotype\nComplexity"] = data["Flagged Advantageous Sites"]
data["P/G Joint\nExceedance"] = data["score"]

dfx = df_ecoselfcontext.loc[
    (df_ecoselfcontext["Root ID"] == 0),
    ["Competition Series","Focal Prevalence", "kind"],
]
dfx["Focal Prevalence"] = dfx["Focal Prevalence"].round()
dfx["target"] = dfx["Competition Series"].isin(dft["Series"])
dfp = dfx.groupby(
    ["Competition Series", "kind", "target"],
).mean().reset_index().pivot(
    columns="kind",
    index=["Competition Series", "target"],
    values="Focal Prevalence",
).reset_index()
dfp["Fitness (No bkgd.)"] = dfp["nobb"]

data = pd.merge(
    data,
    dfp,
    how="left",
    left_on="Series",
    right_on="Competition Series",
)

for y in ["Phenotype\nComplexity", "Genotype\nComplexity", "P/G Joint\nExceedance"]:
    # Calculate statistics
    pearson_r, pearson_p = scipy_stats.pearsonr(data["Fitness (No bkgd.)"], data[y])
    spearman_rho, spearman_p = scipy_stats.spearmanr(data["Fitness (No bkgd.)"], data[y])

    # Format the annotation string
    stats_text = (
        fr"$r={pearson_r:.2f},\rho={spearman_rho:.2f}$"
        + f"\n$p={pearson_p:.3f},{spearman_p:.3f}$"
    )
    print(stats_text)
    with tp.teed(
        sns.regplot,
        data=data,
        y=y,
        x="Fitness (No bkgd.)",
        line_kws={"color": "black", "lw": 0.8},

        teeplot_subdir=teeplot_subdir,
        scatter_kws={"alpha": 0.5, "clip_on": False, "s": 3},
    ) as ax:
        ax.collections[1].set_alpha(0.4)
        sns.despine(ax=ax)
        ax.figure.set_size_inches(2.5, 1.5)
        ax.text(-0.38, -0.1, stats_text,
                transform=ax.transAxes,
                fontsize=5.5,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    dfsx = []
    mwu_results = [] # Store p-values and delta for plotting

    for split in [0.025, 0.05, 0.075, 0.5]:
        split_df = data.copy()
        split_df["split"] = split * 100

        # Identify groups
        is_top = split_df[y] >= split_df[y].quantile(1 - split)

        group_top = split_df.loc[is_top, "Fitness (No bkgd.)"]
        group_bot = split_df.loc[~is_top, "Fitness (No bkgd.)"]

        stat, p_val = scipy_stats.mannwhitneyu(group_top, group_bot, alternative='two-sided')
        d, _ = cliffs_delta(group_top, group_bot) # Calculate Cliff's Delta

        mwu_results.append((p_val, d))

        # Assign string labels
        split_df["top"] = is_top.map({True: "Top X%", False: "Bottom X%"})
        dfsx.append(split_df)

    combined_split_df = pd.concat(dfsx, ignore_index=True)

    with tp.teed(
        sns.boxplot,
        data=combined_split_df,
        x="split",
        y="Fitness (No bkgd.)",
        hue="top",
        notch=True,
        teeplot_subdir=teeplot_subdir,
    ) as ax:
        ax.figure.set_size_inches(2.5, 1.5)
        sns.despine(ax=ax)
        sns.move_legend(
            ax, "lower center",
            bbox_to_anchor=(.35, 1.05), ncol=3, title=None, frameon=False,
        )
        ax.set_xlabel(f"{y} Split (X%)")
        ax.set_ylabel("Fitness\n(No bkgd.)")

        # Determine y-limits to comfortably place annotations below the legend
        y_max = combined_split_df["Fitness (No bkgd.)"].max()
        y_min = combined_split_df["Fitness (No bkgd.)"].min()
        y_range = y_max - y_min
        ax.set_ylim(top=y_max + y_range * 0.15)

        # Annotate Mann-Whitney U test p-values and Cliff's Delta above each split group
        for i, (p_val, d) in enumerate(mwu_results):
            if p_val < 0.05:
                # Format for significance display
                if p_val < 0.001:
                    p_text = "$p < 0.001$"
                else:
                    p_text = f"$p = {p_val:.3f}$"

                # Add Cliff's Delta on a new line
                p_text += f"\n$\\delta={d:.3f}$"

                ax.text(
                    x=i,
                    y=y_max + (y_range * 0.02),
                    s=p_text,
                    ha='center',
                    va='bottom',
                    fontsize=5.5,
                    color='black'
                )

    dfsx = []
    mwu_results = [] # Store p-values and delta for plotting

    for split in [0.025, 0.05, 0.075, 0.5]:
        split_df = data.copy()
        split_df["split"] = split * 100

        # Identify groups
        is_top = split_df["Fitness (No bkgd.)"] >= split_df["Fitness (No bkgd.)"].quantile(1 - split)

        group_top = split_df.loc[is_top, y]
        group_bot = split_df.loc[~is_top, y]

        stat, p_val = scipy_stats.mannwhitneyu(group_top, group_bot, alternative='two-sided')
        d, _ = cliffs_delta(group_top, group_bot) # Calculate Cliff's Delta

        mwu_results.append((p_val, d))

        # Assign string labels
        split_df["top"] = is_top.map({True: "Top X%", False: "Bottom X%"})
        dfsx.append(split_df)

    combined_split_df = pd.concat(dfsx, ignore_index=True)

    with tp.teed(
        sns.boxplot,
        data=combined_split_df,
        x="split",
        y=y,
        hue="top",
        notch=True,
        teeplot_subdir=teeplot_subdir,
    ) as ax:
        ax.figure.set_size_inches(2.5, 1.5)
        sns.despine(ax=ax)
        sns.move_legend(
            ax, "lower center",
            bbox_to_anchor=(.35, 1.05), ncol=3, title=None, frameon=False,
        )
        ax.set_xlabel("Fitness (No bkgd.)\nSplit (X%)")
        ax.set_ylabel(y)

        # Determine y-limits to comfortably place annotations below the legend
        y_max = combined_split_df[y].max()
        y_min = combined_split_df[y].min()
        y_range = y_max - y_min
        ax.set_ylim(top=y_max + y_range * 0.15)

        # Annotate Mann-Whitney U test p-values and Cliff's Delta above each split group
        for i, (p_val, d) in enumerate(mwu_results):
            if p_val < 0.05:
                # Format for significance display
                if p_val < 0.001:
                    p_text = "$p < 0.001$"
                else:
                    p_text = f"$p = {p_val:.3f}$"

                # Add Cliff's Delta on a new line
                p_text += f"\n$\\delta={d:.3f}$"

                ax.text(
                    x=i,
                    y=y_max + (y_range * 0.02),
                    s=p_text,
                    ha='center',
                    va='bottom',
                    fontsize=5.5,
                    color='black'
                )
